# Create z-score files

This notebook creates mean/std files for the paper. Resdidual coefficients are defined in a separated notebook.

In [1]:
import os
import yaml
import numpy as np
import xarray as xr

## ERA5 mean std

In [2]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_ERA5.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [3]:
N_levels = 6

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/ERA5/'
ds_example = xr.open_zarr(base_dir+'ERA5_8km_2000.zarr')
level = np.array(ds_example['level'])

In [4]:
varnames = list(conf['zscore'].keys())
varnames = varnames[:-3] # remove save_loc and others
varname_surf = list(set(varnames) - set(['U', 'V', 'T', 'Q']))
varname_upper = ['U', 'V', 'T', 'Q']

# collect computed mean and variance values
# See "qsub_STEP01_compute_mean_std.ipynb"
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['zscore']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['zscore']['prefix'], varname)
    
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['zscore']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['zscore']['prefix'], i_level, varname)
        
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [5]:
conf['zscore']['save_loc']

'/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_TC/temp_ERA5_npy/'

### Mean file

In [6]:
# ------------------------------------------------------- #
# Initialize dataset
ds_mean = xr.Dataset(coords={"level": level})

for varname, data in MEAN_values.items():
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["level",],
            coords={"level": level},
            name=varname,
        )
        ds_mean[varname] = data_array
    else:
        data_array = xr.DataArray(data, name=varname,)
        ds_mean[varname] = data_array

In [10]:
# ds_mean.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/ERA5_mean_TC_1980_2019.nc', mode='w')
ds_mean = ds_mean.isel(level=slice(1))
ds_mean.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/ERA5_mean_unet_1980_2019.nc', mode='w')

In [11]:
ds_GP = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/ERA5_mean_CorrDiff_1980_2019.nc')
ds_full = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/ERA5_mean_unet_1980_2019.nc')

for varname in ds_full.keys():
    print(f'=================== {varname} ===================')
    try:
        print(ds_GP[varname].values)
        print(ds_full[varname].values)
    except:
        pass

=================== VAR_10U ===================
-0.7922798233896581
-0.7922798233896581
=================== SP ===================
100366.91752960668
100366.91752960668
=================== precip_025 ===================
=================== PWAT_05 ===================
5.064892820938832
5.064892820938832
=================== MSL ===================
101669.14473618966
101669.14473618966
=================== VAR_10V ===================
0.33701485579127577
0.33701485579127577
=================== VAR_2T ===================
292.3129523513315
292.3129523513315
=================== U ===================
[-0.41022275]
[-0.41022275]
=================== V ===================
[1.01054357]
[1.01054357]
=================== T ===================
[289.21060655]
[289.21060655]
=================== Q ===================
[0.00991233]
[0.00991233]


### Std file

In [12]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std = xr.Dataset(coords={"level": level})

for varname, data in STD_values.items():
    data = np.sqrt(data)
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["level",],
            coords={"level": level},
            name=varname,
        )
        ds_std[varname] = data_array
    else:
        data_array = xr.DataArray(data, name=varname)
        ds_std[varname] = data_array

In [13]:
# ds_std.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/ERA5_std_TC_1980_2019.nc', mode='w')
ds_std = ds_std.isel(level=slice(1))
ds_std.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/ERA5_std_unet_1980_2019.nc', mode='w')

In [14]:
ds_GP = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/ERA5_std_CorrDiff_1980_2019.nc')
ds_full = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/ERA5_std_unet_1980_2019.nc')

for varname in ds_full.keys():
    print(f'=================== {varname} ===================')
    try:
        print(ds_GP[varname].values)
        print(ds_full[varname].values)
    except:
        pass

=================== VAR_10U ===================
3.985819368321598
3.985819368321598
=================== SP ===================
1955.694003217757
1955.694003217757
=================== precip_025 ===================
=================== PWAT_05 ===================
1.48523804337076
1.48523804337076
=================== MSL ===================
588.7541706452748
588.7541706452748
=================== VAR_10V ===================
3.837387183428028
3.837387183428028
=================== VAR_2T ===================
9.435816222669837
9.435816222669837
=================== U ===================
[6.06806982]
[6.06806982]
=================== V ===================
[6.06690808]
[6.06690808]
=================== T ===================
[8.57600606]
[8.57600606]
=================== Q ===================
[0.00473722]
[0.00473722]


## WRF mean std

In [13]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_WRF.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [14]:
N_levels = 12

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/C404_8km/'
ds_example = xr.open_zarr(base_dir+'C404_8km_2000.zarr')
level = np.array(ds_example['bottom_top'])

In [15]:
varnames = list(conf['zscore'].keys())
varnames = varnames[:-3] # remove save_loc and others
varname_upper = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_P', 'WRF_Q_tot', 'WRF_Q_tot_05', 'WRF_W', 'WRF_Z']
varname_surf = list(set(varnames) - set(varname_upper))

# collect computed mean and variance values
# See "qsub_STEP01_compute_mean_std.ipynb"
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['zscore']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['zscore']['prefix'], varname)
    
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['zscore']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['zscore']['prefix'], i_level, varname)
        
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [16]:
# ------------------------------------------------------- #
# Initialize dataset
ds_mean = xr.Dataset(coords={'bottom_top': level})

for varname, data in MEAN_values.items():
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_mean[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_mean[varname] = data_array

In [17]:
ds_mean.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/C404_mean_TC_1980_2019_12lev.nc', mode='w')

In [18]:
# ds_new = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_mean_1980_2019_12lev.nc')
# ds_full = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/C404_mean_1980_2019_12lev.nc')

# for varname in ds_full.keys():
#     print(f'=================== {varname} ===================')
#     try:
#         print(ds_full[varname].values)
#         print(ds_new [varname].values)
#     except:
#         pass

In [19]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std = xr.Dataset(coords={'bottom_top': level})

for varname, data in STD_values.items():
    data = np.sqrt(data)
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_std[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std[varname] = data_array

In [20]:
ds_std.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/mean_std/C404_std_TC_1980_2019_12lev.nc', mode='w')

In [21]:
# ds_new = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_std_1980_2019_12lev.nc')
# ds_full = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/C404_std_1980_2019_12lev.nc')

# for varname in ds_full.keys():
#     print(f'=================== {varname} ===================')
#     try:
#         print(ds_full[varname].values)
#         print(ds_new [varname].values)
#     except:
#         pass